# 06_filter — Per-participant stimulus-space filter (pre-image of the fitted distortion)

**Manuscript:** Results section 6; Figure 6; Methods 'Stimulus-space filter and its evaluation'.

For each displayed hue theta_k the filter solves T(theta_pre) = theta_k for the fitted transform T (`exp2_compute_preimage.py`, Brent root finding); the per-hue correction is theta_pre - theta_k. `stim_lab_render.py` renders the corrected discs in CIELab for the PsychoPy display and for Figure 6. The notebook re-applies the forward model to the committed pre-image angles and confirms that every hue resolves to its target within 1e-3 degrees.

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_06_filter.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/sub-0{8,9}_2component_preimage.json` | `scripts/exp2_compute_preimage.py` | fitted parameters, stimulus angles, pre-image angles and per-hue corrections |
| `../04_distortion_model/scripts/two_comp.py` | `(imported)` | forward two-component model used to verify the pre-image |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

sys.path.insert(0, str((Path("..") / "04_distortion_model" / "scripts").resolve()))
from two_comp import forward_2comp, THETA_CONF
P = {"deutan": J("sub-08_2component_preimage.json"), "protan": J("sub-09_2component_preimage.json")}

V.start("06_filter")

### The pre-image filter (Results section 6; Methods 'Stimulus-space filter')
For each displayed hue the filter solves T(theta_pre) = theta; the notebook re-applies the fitted forward model to the committed pre-image angles.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 06.01 | Results §6 | deutan fitted (beta_s, beta_c) | `(6.0, -42.0)` |
| 06.02 | Results §6 | deutan theta_conf | `150.0` |
| 06.03 | Results §6 | deutan mean per-hue correction |delta theta| (degrees) | `26.3` |
| 06.04 | Results §6 | protan fitted (beta_s, beta_c) | `(2.0, 24.0)` |
| 06.05 | Results §6 | protan theta_conf | `16.0` |
| 06.06 | Results §6 | protan mean |delta theta| (degrees) | `16.2` |
| 06.07 | Methods filter | eight hues per participant | `(8, 8)` |
| 06.08 | Methods filter | every hue resolves to within 1e-3 degrees (deutan, recomputed) | `0.001` |
| 06.09 | Methods filter | every hue resolves to within 1e-3 degrees (protan, recomputed) | `0.001` |
| 06.10 | Methods filter | stored per-hue corrections equal pre-image minus stimulus angle | `1e-06` |

In [2]:
res = {}
for k, j in P.items():
    bs, bc = j["phase_a_params"]["beta_s"], j["phase_a_params"]["beta_c"]
    stim = np.array(j["stimulus_angles_cielab"], float); pre = np.array(j["preimage_angles_cielab"], float)
    delta = np.array(j["delta_theta_apply_deg"], float)
    perceived = pre + np.asarray(forward_2comp(bs, bc, j["cvd_type"], hues=pre), float)   # T(theta_pre) = theta_pre + delta(theta_pre)
    resid = np.abs(((perceived - stim) + 180) % 360 - 180)
    res[k] = dict(bs=bs, bc=bc, conf=j["theta_conf_deg"], mean_abs=float(np.mean(np.abs(delta))), max_resid=float(resid.max()), n=len(stim),
                  delta_consistent=float(np.max(np.abs(((pre - stim) + 180) % 360 - 180 - delta))))
    print(k, res[k], np.round(delta, 1))
V.check('06.01', 'Results §6 | deutan fitted (beta_s, beta_c)', (res["deutan"]["bs"], res["deutan"]["bc"]), (6.0, -42.0), mode='eq')
V.check('06.02', 'Results §6 | deutan theta_conf', float(res["deutan"]["conf"]), 150.0, nd=1)
V.check('06.03', 'Results §6 | deutan mean per-hue correction |delta theta| (degrees)', res["deutan"]["mean_abs"], 26.3, nd=1)
V.check('06.04', 'Results §6 | protan fitted (beta_s, beta_c)', (res["protan"]["bs"], res["protan"]["bc"]), (2.0, 24.0), mode='eq')
V.check('06.05', 'Results §6 | protan theta_conf', float(res["protan"]["conf"]), 16.0, nd=1)
V.check('06.06', 'Results §6 | protan mean |delta theta| (degrees)', res["protan"]["mean_abs"], 16.2, nd=1)
V.check('06.07', 'Methods filter | eight hues per participant', (res["deutan"]["n"], res["protan"]["n"]), (8, 8), mode='eq')
V.check('06.08', 'Methods filter | every hue resolves to within 1e-3 degrees (deutan, recomputed)', res["deutan"]["max_resid"], 0.001, mode='lt')
V.check('06.09', 'Methods filter | every hue resolves to within 1e-3 degrees (protan, recomputed)', res["protan"]["max_resid"], 0.001, mode='lt')
V.check('06.10', 'Methods filter | stored per-hue corrections equal pre-image minus stimulus angle', max(res[k]["delta_consistent"] for k in res), 1e-06, mode='lt')

deutan {'bs': 6.0, 'bc': -42.0, 'conf': 150.0, 'mean_abs': 26.283969800549002, 'max_resid': 2.0526158550637774e-10, 'n': 8, 'delta_consistent': 0.0} [-37.9 -32.1  32.   37.9  26.1   9.1  -9.1 -26. ]
protan {'bs': 2.0, 'bc': 24.0, 'conf': 16.0, 'mean_abs': 16.203372945287562, 'max_resid': 1.5336354408646002e-10, 'n': 8, 'delta_consistent': 0.0} [-19.  -24.6 -13.9  16.   24.6  18.1   6.1  -7.3]
[OK ] 06.01 Results §6 | deutan fitted (beta_s, beta_c): produced=(6, -42)  reported=(6, -42)
[OK ] 06.02 Results §6 | deutan theta_conf: produced=150  reported=150
[OK ] 06.03 Results §6 | deutan mean per-hue correction |delta theta| (degrees): produced=26.28  reported=26.3
[OK ] 06.04 Results §6 | protan fitted (beta_s, beta_c): produced=(2, 24)  reported=(2, 24)
[OK ] 06.05 Results §6 | protan theta_conf: produced=16  reported=16
[OK ] 06.06 Results §6 | protan mean |delta theta| (degrees): produced=16.2  reported=16.2
[OK ] 06.07 Methods filter | eight hues per participant: produced=(8, 8)  re

In [3]:
V.summary()


=== 06_filter: 10/10 numeric checks reproduced exactly; 0 within one unit of the last printed digit; 0 mismatch, 0 error, 0 pointer-only ===
